In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

%load_ext autoreload
%autoreload 2
print(os.getcwd())

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/modelling/SurvivEHR/notebooks/CompetingRisk/0_pretraining/3_token_embeddings


In [2]:
import torch
import matplotlib.pyplot as plt
from hydra import compose, initialize
from sklearn.preprocessing import StandardScaler
import umap

from FastEHR.dataloader.foundational_loader import FoundationalDataModule

from SurvivEHR.src.models.survival.task_heads.causal import SurvStreamGPTForCausalModelling
from SurvivEHR.examples.modelling.SurvivEHR.run_experiment import run
from SurvivEHR.examples.data.map_to_reduced_names import EVENT_NAME_SHORT_MAP

torch.manual_seed(1337)
torch.set_float32_matmul_precision('medium')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}.")

 # TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed
%env SLURM_NTASKS_PER_NODE=28   

Using device: cuda.
env: SLURM_NTASKS_PER_NODE=28


In [3]:
# load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../../confs", job_name="testing_notebook"):
    cfg = compose(config_name="config_CompetingRisk11M", 
                  overrides=[# Experiment setup
                             "experiment.type='zeroshot'",
                             "experiment.run_id='SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1'",
                              "experiment.fine_tune_id='CVD-0shot'",
                             "experiment.train=False",
                             "experiment.test=False",
                             'fine_tuning.fine_tune_outcomes=["IHDINCLUDINGMI_OPTIMALV2", "ISCHAEMICSTROKE_V2", "MINFARCTION", "STROKEUNSPECIFIED_V2", "STROKE_HAEMRGIC"]',
                             # Dataloader
                             "data.path_to_ds=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_Hypertension/",
                             "data.meta_information_path=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle",
                             "data.min_workers=12",
                             # "data.batch_size=512",
                             # Optimiser
                             "optim.limit_test_batches=null",
                             # Single-Risk specific
                            ]
                 )     

# cfg.data.path_to_ds="/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_CVD/"
cfg.data.path_to_ds="/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_Hypertension/"


experiment, dm = run(cfg)
print(f"Loaded model with {sum(p.numel() for p in experiment.model.parameters())/1e6} M parameters")


/rds/bear-apps/2022a/EL8-ice/software/PyTorch/2.0.1-foss-2022a-CUDA-11.7.0/lib/python3.10/site-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Loaded model with 11.20919 M parameters


In [8]:
diagnoses = False

In [9]:
token_labels = []
tokens = []

for token, event in enumerate(list(dm.train_set.tokenizer._stoi.keys())):

    # only look at diagnoses
    if diagnoses:
        if event.upper() == event:
            tokens.append(token)
            token_labels.append(event)
    else:
        if event.upper() != event:            
            tokens.append(token)
            token_labels.append(event)
        
        
tokens = torch.tensor(tokens).to(device)
print(tokens.shape)
print(token_labels)

torch.Size([189])
['Plasma_N_terminal_pro_B_type_natriuretic_peptide_conc_70', 'N_terminal_pro_brain_natriuretic_peptide_level_67', 'AllHIVdrugs_HIV', 'Plasma_B_natriuretic_peptide_level_69', 'Plasma_pro_brain_natriuretic_peptide_level_64', 'Albumin___creatinine_ratio_37', 'Urine_microalbumin_creatinine_ratio_36', 'Plasma_ferritin_level_62', 'Brain_natriuretic_peptide_level_66', 'Monoamine_Oxidase_Inhibitors_OPTIMAL', 'Serum_pro_brain_natriuretic_peptide_level_65', 'Serum_vitamin_D2_level_89', 'Total_25_hydroxyvitamin_D_level_91', 'Serum_N_terminal_pro_B_type_natriuretic_peptide_conc_68', 'Blood_calcium_level_38', 'INR___international_normalised_ratio_81', 'Combined_total_vitamin_D2_and_D3_level_93', 'Amantadine', 'Meglitinides_Aurum', 'TSH_level_74', 'Acarbose_AURUM', 'Serum_T4_level_78', 'LeflunomideAURUM_Zhaonan', 'Plasma_cholesterol_HDL_ratio_96', 'Plasma_free_T4_level_77', '25_Hydroxyvitamin_D2_level_92', 'Blood_urea_28', '25_Hydroxyvitamin_D3_level_90', 'MAO_B_Optimal', 'Galantam

In [10]:
tkn_embedding = experiment.model.transformer.wte.dynamic_embedding_layer(tokens)
scaled_tkn_embedding = StandardScaler().fit_transform(tkn_embedding.detach().cpu())

In [11]:
reducer = umap.UMAP()

embedding = reducer.fit_transform(scaled_tkn_embedding)
print(embedding.shape)

plt.figure(figsize=(10,10))
plt.scatter(embedding[:,0], embedding[:, 1], label=token_labels)
for i, txt in enumerate(token_labels):
    txt_short = EVENT_NAME_SHORT_MAP.get(txt, '?')
    plt.annotate(txt_short, (embedding[i,0], embedding[i,1]))

plt.savefig(f"token_embedding_{'diagnosis' if diagnoses else 'other'}.png")
plt.close()

(189, 2)
